## 2. Feature Selection

We select the machine operating and usage features that will be used for failure prediction.

**Input features:**
- Type
- Air temperature
- Process temperature
- Rotational speed
- Torque
- Tool wear

**Target:** Machine failure

Failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are excluded from the initial model because they directly describe failure conditions and may cause target leakage.

In [29]:
import pandas as pd
df=pd.read_csv("../data/raw/ai4i2020.csv")

features=[
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target="Machine failure"

X=df[features]
y=df[target]

X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


## 3. Train-Test Split

The dataset is split into 80% training data and 20% testing data.

Stratification is used because machine failure is highly imbalanced (~3.4% failures). This keeps a similar failure ratio in both sets.

In [30]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training shape: (8000, 6)
Testing shape: (2000, 6)

Training target distribution:
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

Testing target distribution:
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [32]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

## 4. Preprocessing

The categorical `Type` feature is converted using one-hot encoding.

Numerical features are passed through without scaling because the first baseline model will be tree-based.

The preprocessor is fitted only on the training data and then applied to the test data to prevent data leakage.

In [33]:
from src.features.preprocessing import create_preprocessor

preprocessor = create_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [34]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [35]:
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (8000, 8)
Testing shape: (2000, 8)


In [36]:
preprocessor.fit_transform(X_train)

array([[0.000e+00, 0.000e+00, 1.000e+00, ..., 1.456e+03, 4.720e+01,
        5.400e+01],
       [0.000e+00, 0.000e+00, 1.000e+00, ..., 1.399e+03, 4.640e+01,
        1.320e+02],
       [0.000e+00, 0.000e+00, 1.000e+00, ..., 1.357e+03, 4.560e+01,
        1.370e+02],
       ...,
       [1.000e+00, 0.000e+00, 0.000e+00, ..., 1.464e+03, 4.100e+01,
        2.900e+01],
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 1.517e+03, 4.240e+01,
        1.130e+02],
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 1.462e+03, 4.680e+01,
        4.000e+00]], shape=(8000, 8))

In [37]:
preprocessor.transform(X_test)

array([[0.000e+00, 1.000e+00, 0.000e+00, ..., 1.345e+03, 6.270e+01,
        1.530e+02],
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 1.513e+03, 4.010e+01,
        1.350e+02],
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 1.559e+03, 3.760e+01,
        2.090e+02],
       ...,
       [0.000e+00, 0.000e+00, 1.000e+00, ..., 1.446e+03, 4.280e+01,
        1.210e+02],
       [0.000e+00, 0.000e+00, 1.000e+00, ..., 1.422e+03, 4.640e+01,
        1.940e+02],
       [0.000e+00, 0.000e+00, 1.000e+00, ..., 1.970e+03, 2.160e+01,
        3.700e+01]], shape=(2000, 8))

In [38]:
preprocessor.get_feature_names_out()

array(['cat__Type_H', 'cat__Type_L', 'cat__Type_M',
       'num__Air temperature [K]', 'num__Process temperature [K]',
       'num__Rotational speed [rpm]', 'num__Torque [Nm]',
       'num__Tool wear [min]'], dtype=object)

## 5. Preprocessing Result

One categorical feature with 3 categories produces 3 one-hot encoded columns.

Therefore:

5 numerical features + 3 encoded categorical features = 8 features.

In [39]:
X_train_processed[:5]


array([[0.000e+00, 0.000e+00, 1.000e+00, 3.020e+02, 3.109e+02, 1.456e+03,
        4.720e+01, 5.400e+01],
       [0.000e+00, 0.000e+00, 1.000e+00, 2.970e+02, 3.083e+02, 1.399e+03,
        4.640e+01, 1.320e+02],
       [0.000e+00, 0.000e+00, 1.000e+00, 3.010e+02, 3.116e+02, 1.357e+03,
        4.560e+01, 1.370e+02],
       [0.000e+00, 1.000e+00, 0.000e+00, 2.989e+02, 3.098e+02, 1.411e+03,
        5.630e+01, 8.400e+01],
       [0.000e+00, 1.000e+00, 0.000e+00, 2.971e+02, 3.085e+02, 1.733e+03,
        2.870e+01, 5.000e+01]])